## Extract playlist metadata and build a content catalog



In [1]:
!python -V

Python 3.11.5


In [3]:
!pip install yt-dlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 4.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Install dependency
!pip install google-auth-oauthlib google-auth-httplib2


In [3]:
# import required libraries
import pandas as pd
import os
import requests, time
from urllib.parse import urlparse, parse_qs
from yt_dlp import YoutubeDL

In [4]:
# playlist URLs
playlist_urls = [
    "https://www.youtube.com/playlist?list=PLSG9IRx05GOn-ioDWMw5-92PsbPi-MHSH",
    "https://www.youtube.com/playlist?list=PLSG9IRx05GOlqnHGUm2bYVi2KPXYC_lTg"
]

In [5]:
# Output path for the catalog CSV
OUTPUT_CSV = "../data/catalog.csv"

In [6]:
# yt-dlp options (no download)
ydl_opts = {
    "extract_flat": False,   # get full metadata
    "skip_download": True,
    "quiet": True,
    "nocheckcertificate": True,
    "cachedir": False,
    "ignoreerrors": True,
}

In [7]:
def playlist_to_rows(playlist_url):
    rows = []
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(playlist_url, download=False)
        # If the URL is a playlist, info['entries'] is usually the list
        entries = info.get("entries") or []
        for e in entries:
            # Guard: some entries can be None if extraction failed; skip those
            if not e:
                continue
            vid = e.get("id")
            title = (e.get("title") or "").strip()
            url = e.get("webpage_url") or f"https://www.youtube.com/watch?v={vid}"
            duration = e.get("duration")  # seconds or None
            # Detect #Shorts tag at the end of the title (case-insensitive)
            is_short_tag = title.lower().rstrip().endswith("#shorts")
            # If duration is None try to fallback to approximate 'duration' inside 'formats' or 'alt' (rare)
            rows.append({
                "playlist_url": playlist_url,
                "video_id": vid,
                "title": title,
                "url": url,
                "duration_seconds": int(duration) if duration is not None else None,
                "is_short_tag": bool(is_short_tag)
            })
    return rows

In [8]:
# Gather rows from all playlists
all_rows = []
for p in playlist_urls:
    print("Fetching playlist:", p)
    rows = playlist_to_rows(p)
    print(" -> found", len(rows), "entries (including shorts).")
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)


Fetching playlist: https://www.youtube.com/playlist?list=PLSG9IRx05GOn-ioDWMw5-92PsbPi-MHSH


ERROR: [youtube] euq82a4SJdY: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


 -> found 60 entries (including shorts).
Fetching playlist: https://www.youtube.com/playlist?list=PLSG9IRx05GOlqnHGUm2bYVi2KPXYC_lTg


ERROR: [youtube] sr-xcxgPWq8: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] jRlkOU0g9kM: Video unavailable. This video is no longer available due to a copyright claim by Viral DRM


 -> found 61 entries (including shorts).


In [44]:
df.to_csv(OUTPUT_CSV, index=False)

### Best Way To Do

#### Getting the refresh and access token for the Oauth client of the YouTube Data API

In [8]:
CLIENT_ID      = os.getenv("YT_DATA_API_CLIENT_ID")
CLIENT_SECRET  = os.getenv("YT_DATA_API_CLIENT_SECRET")

In [9]:
# getting the refresh token for using the YouTube Data API
from google_auth_oauthlib.flow import InstalledAppFlow
yt_data_api_client_oauth_secret_file = f"../.secrets/client_secret_{CLIENT_ID}.json"
SCOPES = ["https://www.googleapis.com/auth/youtube.readonly"]

flow = InstalledAppFlow.from_client_secrets_file(yt_data_api_client_oauth_secret_file, SCOPES,  redirect_uri='http://localhost:8080/')

auth_url, _ = flow.authorization_url(prompt='consent')
print("Go to this URL and authorize:\n", auth_url)

# After you authorize, you'll get a code in the browser — paste it below
code = input("\nEnter the authorization code here: ").strip()
flow.fetch_token(code=code)

print("\nREFRESH TOKEN:\n", flow.credentials.refresh_token)
print("ACCESS TOKEN:\n", flow.credentials.token)

Go to this URL and authorize:
 https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=556502130617-efkke8hkevbu8a9ciisv5ljj53koihiv.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube.readonly&state=AMyrZ3YtxDziwYFwduH3xTikk2xbRh&prompt=consent&access_type=offline



Enter the authorization code here:  4/0AVGzR1CHHuI6G99hI5F01YqAsUYRmSB6HN8bwPDUNYWuL8U0igLMZdqozDiclChs12VwXg



REFRESH TOKEN:
 1//0gGWJBN9mdhYeCgYIARAAGBASNwF-L9IrsPPIHWSklOo8SspXVbH4FU9GinZICQ99YYdA9c_OU4D4UkuzvAEM559Sn8uBjKA8oWk
ACCESS TOKEN:
 ya29.a0AQQ_BDTqvU7WD7fwLJb7qVDuVUp70ypEqp31lKvn_0Gw-3kl2hFLyj1fUM4mOqVCkpk8gvc1ExSycutLsT_zL_xwjdcTkLfEFtwE79c69G1rOHXP5sdN2BmfP3Yk8FyxkG9GXsIA4WZqlJr6MifaSdvXx9Jq61tjUbtXP3VL3XwveUd5VjRno-2aNxgnrFfiSp_0rmcaCgYKAb0SARESFQHGX2MiVfwHfX-ZWKk75jfnTAnTDQ0206


#### After getting refresh token from above step, set it as environment variable

In [ ]:
REFRESH_TOKEN  = os.getenv("YT_DATA_API_REFRESH_TOKEN")

In [17]:
YT_API = "https://www.googleapis.com/youtube/v3"

def get_access_token():
    r = requests.post("https://oauth2.googleapis.com/token", data={
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": REFRESH_TOKEN,
    })
    r.raise_for_status()
    return r.json()["access_token"]

def iso8601_to_seconds(s):
    if not s:
        return None
    import re
    m = re.findall(r'(\d+)([HMS])', s)
    sec = 0
    for v, u in m:
        v = int(v)
        sec += v * (3600 if u=='H' else 60 if u=='M' else 1)
    return sec

def fetch_playlist_items(token, pid):
    out = []
    token_hdr = {"Authorization": f"Bearer {token}"}
    next_tok = None
    while True:
        p = {"part": "snippet,contentDetails", "playlistId": pid, "maxResults": 50}
        if next_tok: p["pageToken"] = next_tok
        r = requests.get(f"{YT_API}/playlistItems", params=p, headers=token_hdr, timeout=20)
        r.raise_for_status()
        d = r.json()
        out += d.get("items", [])
        next_tok = d.get("nextPageToken")
        if not next_tok:
            break
        time.sleep(0.1)
    return out

def fetch_videos_info(token, vids):
    token_hdr = {"Authorization": f"Bearer {token}"}
    info = {}
    for i in range(0, len(vids), 50):
        batch = vids[i:i+50]
        ids = ",".join(batch)
        resp = requests.get(
            f"{YT_API}/videos",
            params={"part": "contentDetails,snippet", "id": ids},
            headers=token_hdr,
            timeout=20,
        )
        resp.raise_for_status()
        items = resp.json().get("items", [])
        returned = {it["id"] for it in items}
        # record info
        for it in items:
            vid = it["id"]
            dur_iso = it.get("contentDetails", {}).get("duration")
            dur = iso8601_to_seconds(dur_iso) if dur_iso else None
            has_cc = it.get("contentDetails", {}).get("caption", "false").lower() == "true"
            info[vid] = {"duration_seconds": dur, "has_captions": has_cc}
        # warn about missing ids in this batch
        missing = [v for v in batch if v not in returned]
        if missing:
            print(f"Warning: missing metadata for ids (batch {i//50}): {missing}")
        time.sleep(0.1)
    return info

# playlists
playlist_urls = [
    "https://www.youtube.com/playlist?list=PLSG9IRx05GOn-ioDWMw5-92PsbPi-MHSH",
    "https://www.youtube.com/playlist?list=PLSG9IRx05GOlqnHGUm2bYVi2KPXYC_lTg"
]

token = get_access_token()
rows = []

for purl in playlist_urls:
    pid = purl.split("list=")[-1]
    items = fetch_playlist_items(token, pid)
    vids = [i["contentDetails"]["videoId"] for i in items if "contentDetails" in i]
    infos = fetch_videos_info(token, vids)  # dict: vid -> {duration_seconds, has_captions}

    for i in items:
        snip = i.get("snippet", {})
        cd = i.get("contentDetails", {})
        vid = cd.get("videoId")
        if not vid:
            continue
        title = (snip.get("title") or "").strip()
        is_short_tag = title.lower().endswith("#shorts")
        meta = infos.get(vid, {})
        dur = meta.get("duration_seconds")
        has_cc = meta.get("has_captions", False)
        # skip shorts and ultra-short videos
        if is_short_tag or (dur is not None and dur < 60):
            continue
        rows.append({
            "playlist_url": purl,
            "video_id": vid,
            "title": title,
            "url": f"https://www.youtube.com/watch?v={vid}",
            "duration_seconds": dur,
            "is_short_tag": is_short_tag,
            "has_captions": has_cc
        })

df_headless = pd.DataFrame(rows)
print("Videos kept:", len(df_headless))
df_headless.head(10)

Videos kept: 108


,playlist_url,video_id,title,url,duration_seconds,is_short_tag,has_captions
0,https://www.youtube.com/playlist?list=PLSG9IRx...,J3dhgWzddY4,How to Survive Being Locked in a Freezer,https://www.youtube.com/watch?v=J3dhgWzddY4,327.0,False,True
1,https://www.youtube.com/playlist?list=PLSG9IRx...,BhzjcmK0ik8,How to Survive Choking,https://www.youtube.com/watch?v=BhzjcmK0ik8,342.0,False,True
2,https://www.youtube.com/playlist?list=PLSG9IRx...,kRRHRSIBweQ,How to Survive Sun Poisoning,https://www.youtube.com/watch?v=kRRHRSIBweQ,275.0,False,True
3,https://www.youtube.com/playlist?list=PLSG9IRx...,sy205qhJvuI,How to Survive in the Arctic Without Clothes,https://www.youtube.com/watch?v=sy205qhJvuI,329.0,False,False
4,https://www.youtube.com/playlist?list=PLSG9IRx...,k_4UldH-_rc,How to Survive a Breakup,https://www.youtube.com/watch?v=k_4UldH-_rc,351.0,False,True
5,https://www.youtube.com/playlist?list=PLSG9IRx...,euq82a4SJdY,How to Survive Being Kicked in the Groin,https://www.youtube.com/watch?v=euq82a4SJdY,322.0,False,True
6,https://www.youtube.com/playlist?list=PLSG9IRx...,fkQEz8LiHmk,How to Survive the Top 5 Most Deadly Viruses,https://www.youtube.com/watch?v=fkQEz8LiHmk,392.0,False,True
7,https://www.youtube.com/playlist?list=PLSG9IRx...,4lkwF8jx5tY,How to Survive Giving Birth in a Car,https://www.youtube.com/watch?v=4lkwF8jx5tY,288.0,False,True
8,https://www.youtube.com/playlist?list=PLSG9IRx...,rod8bJ0OUNA,How to Survive Walking on Broken Glass,https://www.youtube.com/watch?v=rod8bJ0OUNA,261.0,False,True
9,https://www.youtube.com/playlist?list=PLSG9IRx...,RCnzDBK_tPk,How to Survive Being Bullied Online,https://www.youtube.com/watch?v=RCnzDBK_tPk,372.0,False,True


In [18]:
df_headless.to_csv("../data/content_catalog/content_catalog.csv")

In [20]:
# Quick analysis of your catalog
print("\n=== CATALOG ANALYSIS ===")
print(f"Total videos after filtering: {len(df_headless)}")
print(f"Videos with captions available: {df_headless['has_captions'].sum()}")
print(f"Average duration: {df_headless['duration_seconds'].mean():.1f} seconds")

# Duration distribution
duration_bins = [0, 60, 180, 300, 600, float('inf')]
duration_labels = ['<1min', '1-3min', '3-5min', '5-10min', '10min+']
df_headless['duration_category'] = pd.cut(df_headless['duration_seconds'], bins=duration_bins, labels=duration_labels)
print("\nDuration distribution:")
print(df_headless['duration_category'].value_counts().sort_index())


=== CATALOG ANALYSIS ===
Total videos after filtering: 108
Videos with captions available: 103
Average duration: 378.5 seconds

Duration distribution:
duration_category
<1min       0
1-3min      0
3-5min     29
5-10min    73
10min+      5
Name: count, dtype: int64


#------------------------------------------------------